## Multi-Accent and Multi-Lingual Voice Clone Demo with MeloTTS

In [3]:
!conda create -n openvoice-env python=3.9 -y
!conda activate openvoice-env 
!conda install -c conda-forge transformers=4.30.0 huggingface-hub=0.21.0 tokenizers=0.13.3 -y

# Step 3: Install necessary dependencies for OpenVoice
!conda install -n openvoice librosa==0.10.0 huggingface-hub==0.21.0 tokenizers==0.21.0 scipy==1.13.1 transformers==4.31.0 -y
%cd /home/ec2-user/SageMaker/OpenVoice  # Navigate to the OpenVoice directory
!pip install -e .

Channels:
 - conda-forge
 - nvidia
 - pytorch
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.3.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/openvoice-env

  added / updated specs:
    - python=3.9


The following NEW packages will be INSTALLED:

  _libgcc_mutex      conda-forge/linux-64::_libgcc_mutex-0.1-conda_forge 
  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-2_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-h4bc722e_7 
  ca-certificates    conda-forge/noarch::ca-certificates-2025.1.31-hbd8a1cb_1 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.43-h712a8e2_4 
  libexpat           conda-forge/linux-64::libexpat-2.7.0-h5888daf_0 
  libffi             conda-forge/linux-64::libffi-3.4.6-h2dba641_1 
  libgcc             conda-forge/l

In [4]:
!pip install setuptools==57.5.0
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.3/819.3 kB 33.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 78.1.0
    Uninstalling setuptools-78.1.0:
      Successfully uninstalled setuptools-78.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 146.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 172.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 84.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of matplotlib to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.


In [5]:
!conda install -c conda-forge libsndfile -y
!conda install -c conda-forge openh264 -y

!conda install -c conda-forge python-soundfile -y 

Channels:
 - conda-forge
 - nvidia
 - pytorch
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.3.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/pytorch_p310

  added / updated specs:
    - libsndfile


The following NEW packages will be INSTALLED:

  gettext            conda-forge/linux-64::gettext-0.23.1-h5888daf_0 
  gettext-tools      conda-forge/linux-64::gettext-tools-0.23.1-h5888daf_0 
  libasprintf        conda-forge/linux-64::libasprintf-0.23.1-h8e693c7_0 
  libasprintf-devel  conda-forge/linux-64::libasprintf-devel-0.23.1-h8e693c7_0 
  libflac            conda-forge/linux-64::libflac-1.4.3-h59595ed_0 
  libgettextpo       conda-forge/linux-64::libgettextpo-0.23.1-h5888daf_0 
  libgettextpo-devel conda-forge/linux-64::libgettextpo-devel-0.23.1-h5888daf_0 
  libogg     

In [7]:
!pip install eng_to_ipa==0.0.2
import os
import torch
from openvoice import se_extractor
from openvoice.api import ToneColorConverter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 86.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for eng_to_ipa: filename=eng_to_ipa-0.0.2-py3-none-any.whl size=2822639 sha256=01f3ee9278709a84790359bc8bcbe2b3599df785ed9e0a8dc8d5db114ca03b9c
  Stored in directory: /home/ec2-user/.cache/pip/wheels/5b/ab/07/fe6722f710d8ef8bd0ccb4eb689ef96f5552f3fc0c80c1aa9c
Successfully built eng_to_ipa


In [14]:
import zipfile
import os

def unzip_file(zip_path, extract_to='.'):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"Extracted all files to: {os.path.abspath(extract_to)}")

# Example usage:
unzip_file('/home/ec2-user/SageMaker/OpenVoice/checkpoints_v2_0417.zip', '/home/ec2-user/SageMaker/OpenVoice/checkpoints_v2/')

Extracted all files to: /home/ec2-user/SageMaker/OpenVoice/checkpoints_v2


### Initialization

In this example, we will use the checkpoints from OpenVoiceV2. OpenVoiceV2 is trained with more aggressive augmentations and thus demonstrate better robustness in some cases.

In [16]:
ckpt_converter = '/home/ec2-user/SageMaker/OpenVoice/checkpoints_v2/checkpoints_v2/converter'
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = 'outputs_v2'

tone_color_converter = ToneColorConverter(f'{ckpt_converter}/config.json', device=device)
tone_color_converter.load_ckpt(f'{ckpt_converter}/checkpoint.pth')

os.makedirs(output_dir, exist_ok=True)

Loaded checkpoint '/home/ec2-user/SageMaker/OpenVoice/checkpoints_v2/checkpoints_v2/converter/checkpoint.pth'
missing/unexpected keys: [] []


### Obtain Tone Color Embedding
We only extract the tone color embedding for the target speaker. The source tone color embeddings can be directly loaded from `checkpoints_v2/ses` folder.

In [17]:

reference_speaker = 'resources/priyanka_voice.mp3' # This is the voice you want to clone
target_se, audio_name = se_extractor.get_se(reference_speaker, tone_color_converter, vad=True)

OpenVoice version: v2


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/hub.py:294: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/ec2-user/.cache/torch/hub/master.zip


[(0.302, 4.754), (4.782, 17.33), (17.71, 24.658), (24.878, 59.304)]
after vad: dur = 58.374


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/functional.py:660: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at /opt/conda/conda-bld/pytorch_1711403380909/work/aten/src/ATen/native/SpectralOps.cpp:874.)
  return _VF.stft(input, n_fft, hop_length, win_length, window,  # type: ignore[attr-defined]


In [34]:
!sudo chmod u+w /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/melo/text/japanese.py
!~/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/melo/text/japanese.py



/bin/sh: /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/melo/text/japanese.py: Permission denied


#### Use MeloTTS as Base Speakers

MeloTTS is a high-quality multi-lingual text-to-speech library by @MyShell.ai, supporting languages including English (American, British, Indian, Australian, Default), Spanish, French, Chinese, Japanese, Korean. In the following example, we will use the models in MeloTTS as the base speakers. 

In [41]:
!pip install git+https://github.com/myshell-ai/MeloTTS.git
!pip install unidic
!python -m unidic.download

# Monkey-patch MeCab BEFORE importing melo

import MeCab
import unidic


# Only patch if not already patched
if not hasattr(MeCab, '_original_Tagger'):
    MeCab._original_Tagger = MeCab.Tagger
    MeCab.Tagger = lambda args='': MeCab._original_Tagger(f"-d {unidic.DICDIR}")

import importlib
import MeCab

# This resets the module completely
importlib.reload(MeCab)



from melo.api import TTS


texts = {
    'EN_NEWEST': "Did you ever hear a folk tale about a giant turtle?",  # The newest English base speaker model
    'EN': "Did you ever hear a folk tale about a giant turtle?",
    'ES': "El resplandor del sol acaricia las olas, pintando el cielo con una paleta deslumbrante.",
    #'FR': "La lueur dorée du soleil caresse les vagues, peignant le ciel d'une palette éblouissante.",
    #'ZH': "在这次vacation中，我们计划去Paris欣赏埃菲尔铁塔和卢浮宫的美景。",
    
    #'KR': "안녕하세요! 오늘은 날씨가 정말 좋네요.",
}


src_path = f'{output_dir}/tmp.wav'

# Speed is adjustable
speed = 1.0

for language, text in texts.items():
    model = TTS(language=language, device=device)
    speaker_ids = model.hps.data.spk2id
    
    for speaker_key in speaker_ids.keys():
        speaker_id = speaker_ids[speaker_key]
        speaker_key = speaker_key.lower().replace('_', '-')
        
        source_se = torch.load(f'checkpoints_v2/base_speakers/ses/{speaker_key}.pth', map_location=device)
        if torch.backends.mps.is_available() and device == 'cpu':
            torch.backends.mps.is_available = lambda: False
        model.tts_to_file(text, speaker_id, src_path, speed=speed)
        save_path = f'{output_dir}/output_v2_{speaker_key}.wav'

        # Run the tone color converter
        encode_message = "@MyShell"
        tone_color_converter.convert(
            audio_src_path=src_path, 
            src_se=source_se, 
            tgt_se=target_se, 
            output_path=save_path,
            message=encode_message)
from IPython.display import Audio, display
print("original:")
display(Audio(filename=reference_speaker))
print("Base TTS Output:")
display(Audio(filename=src_path))

print("Tone Converted Output:")
display(Audio(filename=save_path))

  Cloning https://github.com/myshell-ai/MeloTTS.git to /tmp/pip-req-build-o4qtry3q
  Running command git clone --filter=blob:none --quiet https://github.com/myshell-ai/MeloTTS.git /tmp/pip-req-build-o4qtry3q
  Resolved https://github.com/myshell-ai/MeloTTS.git to commit 209145371cff8fc3bd60d7be902ea69cbdb7965a
  Preparing metadata (setup.py) ... done


RuntimeError: 
----------------------------------------------------------

Failed initializing MeCab. Please see the README for possible solutions:

    https://github.com/SamuraiT/mecab-python3#common-issues

If you are still having trouble, please file an issue here, and include the
ERROR DETAILS below:

    https://github.com/SamuraiT/mecab-python3/issues

issueを英語で書く必要はありません。

------------------- ERROR DETAILS ------------------------
arguments: 
default dictionary path: /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/unidic/dicdir
[ifs] no such file or directory: /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/unidic/dicdir/mecabrc
----------------------------------------------------------
